<a href="https://www.kaggle.com/code/lalitacanada/baseball-statistical-modeling-linear-regression?scriptVersionId=354779398" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Baseball Statistical Modeling — Linear Regression
## York University — Big Data Analytics Certificate (2024)

**Author:** Lalita  
**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn  
**Dataset:** Moneyball MLB Statistics 1962-2012 (1,232 records)  

---

## Project Overview
This project applies Linear Regression to predict MLB team wins using batting and pitching statistics from the Moneyball dataset, covering 50 years of Major League Baseball.

## Model Performance
| Metric | Score |
|---|---|
| MAE | 3.24 wins |
| RMSE | 4.04 wins |
| R² Score | 0.868 (explains 86.8% of the variation in wins) |

## Key Findings
- Runs Allowed (RA) has the strongest effect on wins
- Runs Scored (RS) is the second strongest driver
- After standardizing features, OBP, SLG, and BA add little once runs are included
- Predictions are off by about 3 wins on average

## Conclusion
Run differential is the key to winning. Features were standardized before comparing coefficients, because runs and percentages are measured on very different scales.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

df = pd.read_csv('/kaggle/input/datasets/wduckett/moneyball-mlb-stats-19622012/baseball.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print(df.head())

# Drop columns with too many missing values (not used in the model)
df_clean = df.drop(columns=['RankSeason', 'RankPlayoffs', 'OOBP', 'OSLG'])

print("Cleaned Shape:", df_clean.shape)
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

# Runs Scored vs Wins scatter plot
plt.figure(figsize=(10,6))
sns.scatterplot(data=df_clean, 
                x='RS', 
                y='W',
                color='blue',
                alpha=0.5)
plt.title('Runs Scored vs Wins — MLB 1962-2012')
plt.xlabel('Runs Scored (RS)')
plt.ylabel('Wins (W)')
plt.show()

# Features and target
X = df_clean[['RS', 'RA', 'OBP', 'SLG', 'BA']]
y = df_clean['W']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Build Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluate model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("=== MODEL PERFORMANCE ===")
print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {model.score(X_test, y_test):.4f}")

plt.figure(figsize=(10,6))
plt.scatter(y_test, y_pred, 
            color='blue', 
            alpha=0.5)
plt.plot([y_test.min(), y_test.max()], 
         [y_test.min(), y_test.max()], 
         'r--', linewidth=2)
plt.title('Actual vs Predicted Wins — Linear Regression')
plt.xlabel('Actual Wins')
plt.ylabel('Predicted Wins')
plt.show()

from sklearn.preprocessing import StandardScaler

# Features are on very different scales (runs vs. percentages),
# so standardize them before comparing coefficients
X_scaled = StandardScaler().fit_transform(X_train)
model_std = LinearRegression().fit(X_scaled, y_train)

coefficients = pd.Series(model_std.coef_, index=X.columns)
print("Standardized coefficients (wins per 1 standard deviation):")
print(coefficients.sort_values(ascending=False).round(2))

plt.figure(figsize=(10,6))
coefficients.sort_values().plot(kind='barh', color='steelblue')
plt.title('Feature Importance — Standardized Coefficients')
plt.xlabel('Change in Wins per 1 Standard Deviation')
plt.axvline(x=0, color='red', linestyle='--')
plt.show()

print("=" * 50)
print("KEY INSIGHTS — Baseball Statistical Modeling")
print("=" * 50)

print("\nDataset Overview:")
print(f"Total records analyzed: {len(df_clean)}")
print("Years covered: 1962 — 2012")
print("Features used: RS, RA, OBP, SLG, BA")

print("\nModel Performance:")
print(f"MAE:      {mae:.2f} wins")
print(f"RMSE:     {rmse:.2f} wins")
print(f"R2 Score: {model.score(X_test, y_test):.4f} (explains 86.8% of the variation in wins)")

print("\nKey Findings:")
print("1. Runs Allowed (RA) has the strongest effect on wins — preventing runs matters most")
print("2. Runs Scored (RS) is the second strongest driver of wins")
print("3. Once runs are in the model, OBP, SLG, and BA add very little,")
print("   because they mainly affect wins by producing runs")
print("4. Predictions are off by about 3 wins on average (MAE 3.24)")

print("\nConclusion:")
print("Linear Regression explains 86.8% of the variation in MLB team wins.")
print("Run differential (runs scored minus runs allowed) is the key to winning.")